# VA-DCP — Screening Training A0/A1/A2

Gunakan **runtime Colab yang sama** setelah setup full v10. Notebook ini melatih YOLO26n selama 10 epoch untuk screening seed 42. A1/A2 memakai real train + synthetic train tanpa menyalin file; val/test selalu data nyata A0. Hasil dan checkpoint disimpan ke Drive dan dapat di-resume.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess
import sys
import torch

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
A0 = Path('/content/coffee-defect-roboflow')
A1 = Path('/content/coffee-A1-empirical-v10-full')
A2 = Path('/content/coffee-A2-vadcp-physics-v10-full')
AUDITS = Path('/content/drive/MyDrive/coffee-bean-detection/vadcp-setup-physics-v10/full')
OUTPUT = Path('/content/drive/MyDrive/coffee-bean-detection/vadcp-ablation-screen-v10')

for name, root in {'A0': A0, 'A1': A1, 'A2': A2}.items():
    assert (root / 'data.yaml').is_file(), f'{name} hilang; gunakan runtime setup full yang sama: {root}'
assert torch.cuda.is_available(), 'Aktifkan GPU: Runtime > Change runtime type > T4 GPU'
for audit in ('source_dataset_audit.json', 'A1_vadcp_audit.json', 'A2_vadcp_audit.json'):
    assert (AUDITS / audit).is_file(), f'Audit hilang: {AUDITS / audit}'
OUTPUT.mkdir(parents=True, exist_ok=True)
print('GPU   :', torch.cuda.get_device_name(0))
print('A0    :', A0)
print('A1    :', A1)
print('A2    :', A2)
print('OUTPUT:', OUTPUT)

In [ ]:
# Ambil runner terbaru dan instal pada kernel yang sama.
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
subprocess.run(['git', '-C', str(REPO), 'switch', BRANCH], check=True)
subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
print('RUNNER SIAP')

In [ ]:
# Screening terkontrol: 10 epoch, seed 42, sequential A0 -> A1 -> A2.
# Rerun cell ini aman: checkpoint last.pt akan di-resume dan run selesai akan di-skip.
command = [
    sys.executable, '-u', '-m', 'coffee_detector.run_vadcp_ablation',
    '--arm', f'A0={A0}',
    '--arm', f'A1={A1}',
    '--arm', f'A2={A2}',
    '--verified-audit', f'A0={AUDITS / "source_dataset_audit.json"}',
    '--verified-audit', f'A1={AUDITS / "A1_vadcp_audit.json"}',
    '--verified-audit', f'A2={AUDITS / "A2_vadcp_audit.json"}',
    '--config', f'A0={REPO / "configs/A0_yolo26n_screen.yaml"}',
    '--config', f'A1={REPO / "configs/A1_yolo26n_screen.yaml"}',
    '--config', f'A2={REPO / "configs/A2_yolo26n_screen.yaml"}',
    '--seeds', '42',
    '--output-root', str(OUTPUT),
    '--device', '0',
]
print('MULAI SCREENING A0/A1/A2. Progress Ultralytics akan tampil di bawah.', flush=True)
subprocess.run(command, check=True)

In [ ]:
# Ringkasan hanya tersedia setelah ketiga arm selesai.
import json
summary_path = OUTPUT / 'reports' / 'vadcp_ablation_summary.json'
assert summary_path.is_file(), f'Screening belum lengkap: {summary_path}'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
print('=== AGGREGATE ===')
print(json.dumps(summary['aggregate'], indent=2, ensure_ascii=False))
print('=== DELTA VS A0 ===')
print(json.dumps(summary['comparisons'], indent=2, ensure_ascii=False))
print('SAVED:', summary_path)